In [2]:
#setup Stuff

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime, timezone

with open("adData/ads_viewed.json") as f:
    adData = json.load(f)

In [ ]:
#dump

print (adData)

#keysI care Abt
# timestamp, media, label_values, owner

time = [entry["timestamp"] for entry in adData]

#looked for a way to translate the timestamp
from datetime import datetime
for ts in time:
    datum = datetime.utcfromtimestamp(ts).strftime("%d.%m.%Y")
    print(datum)

[{'timestamp': 1773602071, 'media': [], 'label_values': [{'label': 'Ad library public URL'}, {'label': 'URL', 'value': 'https://www.instagram.com/p/DUDde-AgEmr/', 'href': 'https://www.instagram.com/p/DUDde-AgEmr/'}, {'dict': [{'dict': [{'label': 'URL', 'value': 'https://linkin.bio/dedarmilano'}, {'label': 'Name', 'value': 'Dedar'}, {'label': 'Username', 'value': 'dedarmilano'}], 'title': ''}], 'title': 'Owner'}], 'fbid': '17841402133503003:17933582778079939:1773602071::DYIImpressionsData'}, {'timestamp': 1773617686, 'media': [], 'label_values': [{'label': 'Ad library public URL'}, {'label': 'URL', 'value': 'https://www.instagram.com/p/DVeioVdAJqa/', 'href': 'https://www.instagram.com/p/DVeioVdAJqa/'}, {'dict': [{'dict': [{'label': 'URL', 'value': 'http://linkin.bio/birdygrey'}, {'label': 'Name', 'value': 'BIRDY GREY ð\x9f\x90¥'}, {'label': 'Username', 'value': 'birdygrey'}], 'title': ''}], 'title': 'Owner'}], 'fbid': '17841402133503003:17912593839337223:1773617686::DYIImpressionsData'}

In [13]:
#-> dataframe

records = []
for entry in adData:
    timeStamp = entry["timestamp"]

    #i was lost as to how to translate the time a lil so i asked chatGPT
    dtime = datetime.fromtimestamp(timeStamp, tz=timezone.utc)

   
    name = None
    for item in entry["label_values"]:
        if item.get("title") == "Owner":
            for d in item["dict"]:
                for field in d["dict"]:
                    if field["label"] == "Name":
                        name = field["value"]
    
    records.append({"datetime": dtime, "hour": dtime.hour, "advertiser": name})

df = pd.DataFrame(records)


hourly = df.groupby("hour").size().reindex(range(24), fill_value=0)
print(hourly)

hour
0      2
1      1
2     16
3      7
4      2
5      1
6      0
7      1
8      1
9      0
10     0
11     0
12     2
13     1
14     1
15     0
16     7
17     0
18     1
19     3
20     0
21     0
22     1
23     5
dtype: int64


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(hourly.index, hourly.values, color="#EBCDD7", width=0.7, zorder=2)
ax.set_xlabel("Hour", fontsize=11,color = "#778776")
ax.set_ylabel("Ads", fontsize=11,color = "#778776" )
ax.set_title("Time of Activity", fontsize=13, fontweight="bold", pad=14, color = "#778776")

hour_labels = [f"{h}:00" for h in range(24)]
ax.set_xticks(range(24))
ax.set_xticklabels(hour_labels, rotation=45, ha="right", fontsize=8,color = "#778776")


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))  

ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=1)

plt.tight_layout()
plt.savefig("ad_impressions_by_hour.png", dpi=150)
plt.show()

AttributeError: Figure.set() got an unexpected keyword argument 'color'

**Reflection**

On the work itself, I iterated the for loop to get the value I wanted from the data and cleaned it up using some help from AI, but the actual formatting itself was really simple with data frame. The same goes for matplotlib,,, the more arduous part was just finding and learning the correct syntax. I referenced some sample codes but I would say I'm still quite unfamilar with it.... The customization reminds me of css though, and it was fun to play around with.

Seeing the result from the data munging was really fun though. I was first really surprised at how small the data was, so one of the first thing I did was translate the timestamp to actual formatted time and it turns out that the ads were all from January 17th to January 18th.
In my first reflection, I mentioned that I didn't find data inherently malevolent, but looking through the ads I receive, I did end up purchasing or using certain services from these ads (or at the very least, vising the stores), which is a bit eerie.



